<a href="https://colab.research.google.com/github/knc6/jarvis-tools-notebooks/blob/master/jarvis-tools-notebooks/cross_model_comaprison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cross model coparisons for data in [JARVIS-Leaderbaord](https://pages.nist.gov/jarvis_leaderboard/)

# Table of contents

1. Install/download JARVIS-Leaderboard
2. Example of catalysis


In [ ]:
import os
if not os.path.exists('jarvis_leaderboard'):
  !git clone https://github.com/usnistgov/jarvis_leaderboard.git
os.chdir('jarvis_leaderboard')
!pip install -e .

In [ ]:
pip install alignn

In [ ]:
from alignn.graphs import Graph
from alignn.pretrained import get_figshare_model
from tqdm import tqdm
from jarvis.core.atoms import Atoms
from jarvis.db.figshare import data
import pandas as pd
from sklearn.metrics import mean_absolute_error
from jarvis_leaderboard.rebuild import get_metric_value,get_results

res = get_metric_value(csv_path="jarvis_leaderboard/contributions/alignn_model/AI-SinglePropertyPrediction-ead-AGRA_CO-test-mae.csv.zip")
models=['tinnet_O_alignn','tinnet_N_alignn','tinnet_OH_alignn','AGRA_O_alignn','AGRA_OH_alignn','AGRA_CHO_alignn','AGRA_CO_alignn','AGRA_COOH_alignn','ocp2020_all']
benchmarks = ['AI-SinglePropertyPrediction-ead-AGRA_CO-test-mae.csv.zip','AI-SinglePropertyPrediction-ead-AGRA_O-test-mae.csv.zip','AI-SinglePropertyPrediction-ead-AGRA_OH-test-mae.csv.zip','AI-SinglePropertyPrediction-ead-tinnet_N-test-mae.csv.zip','AI-SinglePropertyPrediction-ead-tinnet_O-test-mae.csv.zip','AI-SinglePropertyPrediction-ead-tinnet_OH-test-mae.csv.zip','AI-SinglePropertyPrediction-relaxed_energy-ocp_all-test-mae.csv.zip']

mem=[]
id_tag='id'
for ib in benchmarks:
  bn = os.path.abspath(os.path.join("jarvis_leaderboard","contributions","alignn_model",ib))
  res = get_metric_value(csv_path=bn)
  for jm in models:
    model=get_figshare_model(jm)
    dataset=ib.split('-')[3]
    prop=ib.split('-')[2]
    dat=pd.DataFrame(data(dataset))
    pd_merged = pd.merge(res['df'],dat,on=id_tag)

    x=[]
    y=[]
    z=[]
    for i,ii in tqdm(pd_merged.iterrows()):
      id=ii['id']
      g,lg=Graph.atom_dgl_multigraph(Atoms.from_dict(ii['atoms']))

      pred=model([g,lg]).detach().cpu().numpy().tolist()

      #print(id,pred,ii['prediction'],ii['target'])
      x.append(ii['target'])
      y.append(ii['prediction'])
      z.append(pred)
    mae_old = mean_absolute_error(x,y)
    mae_new = mean_absolute_error(x,z)
    info={}
    info['model']=jm
    info['bench']=ib
    info['len']=len(x)
    info['mae_old']=mae_old
    info['mae_new']=mae_new
    mem.append(info)
    print(mem[-1])

  #break

In [ ]:

mean_absolute_error(x,y),mean_absolute_error(x,z)